# 06 — Drone-Based Beam Mapping: Simulation & Fit

End-to-end workflow:
1. Generate a synthetic zigzag drone flight path.
2. Evaluate a Gaussian beam (and a Zernike-perturbed beam) at the scattered track coordinates.
3. Recover the beam parameters by fitting `GaussianPathFit` + `ZernikePathFit`.
4. Compare true vs recovered beam on a dense grid.

**No external data needed.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from beamlab21.drone import (
    create_drone_path,
    evaluate_gaussian_on_path,
    fit_beam_on_path,
)

## 1 · Generate a drone flight path

In [ ]:
# Beam covers roughly 0–120° (centred at 60°, σ ≈ 15°)
# The zigzag path scans the full field of view
coords_ew = create_drone_path(
    width=120.0, height=120.0,
    dx=4.0, dy=4.0,     # scan-line spacing [deg]
    ds=0.5,              # sample spacing along track [deg]
    jitter=0.3,          # positional scatter [deg]
    direction="EW",
)

coords_ns = create_drone_path(
    width=120.0, height=120.0,
    dx=4.0, dy=4.0,
    ds=0.5,
    jitter=0.3,
    direction="NS",
)

print(f"EW path: {len(coords_ew):,} samples")
print(f"NS path: {len(coords_ns):,} samples")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, coords, label in zip(axes, [coords_ew, coords_ns], ["EW", "NS"]):
    ax.scatter(coords[:, 0], coords[:, 1], s=0.3, c=np.arange(len(coords)),
               cmap="viridis", rasterized=True)
    ax.set_aspect("equal")
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    ax.set_title(f"{label} scan path  ({len(coords):,} pts)")

plt.tight_layout()
plt.show()

## 2 · Evaluate a Gaussian beam along the path

The beam is centred at **(60°, 60°)** — the centre of the scan window.

In [ ]:
rng = np.random.default_rng(42)

# True parameters: [amp, sigx, sigy, xo, yo, tilt_deg]
TRUE_PARAMS = [1.0, 15.0, 13.0, 60.0, 60.0, 8.0]

# Sample from EW path with a small noise floor
x_obs = coords_ew[:, 0]
y_obs = coords_ew[:, 1]

signal_clean = evaluate_gaussian_on_path(x_obs, y_obs, TRUE_PARAMS)
noise        = rng.normal(0, 0.01, len(signal_clean))
signal_noisy = np.clip(signal_clean + noise, 0, None)

print(f"Signal range: {signal_noisy.min():.4f} – {signal_noisy.max():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(x_obs, y_obs, c=signal_noisy, s=0.5, cmap="inferno",
                vmin=0, vmax=1, rasterized=True)
plt.colorbar(sc, ax=ax, label="Signal (normalised)")
ax.set_aspect("equal")
ax.set_xlabel("x [deg]")
ax.set_ylabel("y [deg]")
ax.set_title("Synthetic drone observation (EW path)")
ax.plot(*TRUE_PARAMS[3:5], "w+", markersize=12, markeredgewidth=2, label="True centre")
ax.legend()
plt.tight_layout()
plt.show()

## 3 · Fit: Gaussian + Zernike on the scattered track

In [ ]:
result = fit_beam_on_path(
    x_obs, y_obs, signal_noisy,
    N=10,     # number of Zernike modes
    fac=3.0,  # scale-parameter shrink factor
)

gp = result.gaussian.params
print("\nGaussian fit:")
print(f"  amp  = {gp[0]:.4f}   (true {TRUE_PARAMS[0]})")
print(f"  sigx = {gp[1]:.3f}°  (true {TRUE_PARAMS[1]}°)")
print(f"  sigy = {gp[2]:.3f}°  (true {TRUE_PARAMS[2]}°)")
print(f"  xo   = {gp[3]:.3f}°  (true {TRUE_PARAMS[3]}°)")
print(f"  yo   = {gp[4]:.3f}°  (true {TRUE_PARAMS[4]}°)")
print(f"  tilt = {gp[5]:.3f}°  (true {TRUE_PARAMS[5]}°)")
print(f"\nZernike χ²  = {result.zernike.chi2:.5f}")
print(f"N coeff     = {len(result.zernike.coef)}")

## 4 · Reconstruct the beam model on a dense grid and compare

In [ ]:
from beamlab21.drone import evaluate_zernike_on_path
from beamlab21.models import twoD_Gaussian

# Dense grid centred on the beam
xy_max_grid = 40.0
cx, cy = TRUE_PARAMS[3], TRUE_PARAMS[4]
xg = np.linspace(cx - xy_max_grid, cx + xy_max_grid, 200)
yg = np.linspace(cy - xy_max_grid, cy + xy_max_grid, 200)

# True beam on grid
truth_grid = twoD_Gaussian(xg, yg, TRUE_PARAMS)

# Fitted Gaussian on grid
fit_gauss_grid = twoD_Gaussian(xg, yg, result.gaussian.params)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
kw  = dict(cmap="inferno", vmin=0, vmax=1)
rkw = dict(cmap="RdBu_r",  vmin=-0.05, vmax=0.05)

axes[0].pcolormesh(xg, yg, truth_grid / truth_grid.max(), **kw)
axes[0].set_title("True beam")

axes[1].pcolormesh(xg, yg, fit_gauss_grid / fit_gauss_grid.max(), **kw)
axes[1].set_title("Fitted Gaussian")

resid = (truth_grid / truth_grid.max()) - (fit_gauss_grid / fit_gauss_grid.max())
im = axes[2].pcolormesh(xg, yg, resid, **rkw)
plt.colorbar(im, ax=axes[2], label="residual")
axes[2].set_title("Residual (truth − Gaussian fit)")

for ax in axes:
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    ax.plot(cx, cy, "w+", markersize=10, markeredgewidth=1.5)

plt.tight_layout()
plt.show()

## 5 · Zernike coefficients recovered from the path

In [ ]:
from beamlab21.zernike import NollToQuantum

coef = result.zernike.coef
j_idx = np.arange(1, len(coef) + 1)
labels = [f"Z{j}" for j in j_idx]

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(j_idx, coef, color="steelblue", edgecolor="k", linewidth=0.5)
ax.set_xticks(j_idx)
ax.set_xticklabels(labels, fontsize=9)
ax.axhline(0, color="k", linewidth=0.8)
ax.set_xlabel("Noll index j")
ax.set_ylabel("Coefficient")
ax.set_title("Zernike coefficients recovered from drone track")
plt.tight_layout()
plt.show()

## 6 · NS vs EW path coverage comparison

In [ ]:
sig_ns = evaluate_gaussian_on_path(
    coords_ns[:, 0], coords_ns[:, 1], TRUE_PARAMS
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, coords, sig, label in zip(
    axes,
    [coords_ew, coords_ns],
    [signal_noisy, sig_ns],
    ["EW scan", "NS scan"]
):
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=sig,
                    s=0.4, cmap="inferno", vmin=0, vmax=1, rasterized=True)
    plt.colorbar(sc, ax=ax, label="Signal")
    ax.set_aspect("equal")
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
    ax.set_title(label)
    ax.plot(cx, cy, "w+", markersize=12, markeredgewidth=2)

plt.suptitle("Drone path coverage — EW vs NS scan directions", y=1.02)
plt.tight_layout()
plt.show()